### Exploration
In this notebook, we'll explore the current limitations of tiny language models for mathematical reasoning. We'll start by experimenting with a 0.5B parameter model, running it on a variety of math questions to better understand its level of mathematical ability, discover its weaknesses, and analyze where it succeeds and where it fails.

#### Loading the model

In [15]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))



from transformers import AutoModelForCausalLM, AutoTokenizer


model_str = "Qwen/Qwen3-4B-Base"  # base model
# Where to cache downloaded models (relative to cwd when notebook runs)
CACHE_DIR = Path("models")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_str, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(model_str, dtype="bfloat16", device_map="auto", cache_dir=CACHE_DIR)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 504.60it/s, Materializing param=model.norm.weight]                              


### Below is the format used in the [**DeepSeek R1 paper**](https://arxiv.org/abs/2501.12948)

<div style="font-size: 0.85em; max-width: 750px; line-height: 1.5;">

---
*A conversation between User and Assistant. The user asks a question, and the Assistant solves 
it. The assistant first thinks about the reasoning process in the mind and then provides the user 
with the answer. The reasoning process and answer are enclosed within \<think>...\</think> 
and \<answer>...\</answer> tags, respectively, i.e., \<think> reasoning process here \</think> 
\<answer> answer here \</answer>. User: <span style="color:red">prompt</span>. Assistant:*

---

</div>

We are working with a base model, which is why we have to specify that this is a conversation between a User and Assistant, as well as have the ending be **Assistant:**, because the model will try to predict what an assistant would respond in such a scenario. 

We can create a function for generating this "system prompt" for an arbitrary question, by replacing the <span style="color:red">prompt</span>.

In [57]:
def generate_prompt(question, helper=""):
    """
    Wraps a question into the DeepSeek-R1 paper prompt format.
    """

    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        "The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, "
        "respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. "
        f"User: {question}. Assistant: {helper}"
    )
    return prompt

We can now use the model with the "system" prompt, and try to get it to answer a math question.

Most likely, the model will completely fail at using the specified tags like it should, even with the added `\<think>` helper token we added.

In [58]:
from utils import lmprint

question = "what is 14 times 5670?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=1000,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.7,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│   14 times 5670 equals 79,380. To find the answer, I first multiply 14 by 5670: 14 * 5670 = 79,380.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 💡 Answer ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  79,380                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw output: A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: what is 14 times 5670?. Assistant: <think> 14 times 5670 equals 79,380. To find the answer, I first multiply 14 by 5670: 14 * 5670 = 79,380. <answer>79,380</answer><|endoftext|>


If the model cannot even follow along with the tags that we specified, perhaps it will not be possible to even get it to start improving, because it will always get bad rewards, and never be able to start improving.

Lets try to give our model some chances at this, because it only has to do it a couple of times for each rollout, since we are going to be doing multiple generations per question. 
Eventually it should learn to do this very consistently.

In [ ]:
from utils import checks

question = "what is 14 times 5670?"
helper = "<think>"
prompt = generate_prompt(question, helper=helper)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]
num_tries = 2



inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

outputs = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.8,        # Adds randomness
                num_return_sequences=num_tries, # This does the parallel work for n tries
                pad_token_id=tokenizer.eos_token_id
            )

answer_count = 0
think_count = 0
for i in range(num_tries):
    generated_tokens = outputs[i][input_len:]
    raw_response = helper + tokenizer.decode(generated_tokens, skip_special_tokens=True) # Add back helper

    if checks.has_complete_answer_block(raw_response): answer_count += 1
    if checks.has_complete_thinking_block(raw_response): answer_count += 1
    
    if checks.is_format_correct(raw_response):
        lmprint.pretty_print(raw_response)

    print(raw_response)

print(f"Thinking: {(think_count/num_tries)*100:.1f}% of tries.\n",
      f"Answer: {(answer_count/num_tries)*100:.1f}%")
    

<think> reasoning process here  weit 14 times 5670 is calculated by multiplying the two numbers together. 14 multiplied by 5670 equals 79,380.
<answer>79,380</answer>
<think> reasoning process here הוצאת reasoning process here Here's my thought process:

To solve this problem, I'll simply multiply the two numbers together. 

14 × 5670 = (10 + 4) × 5670
= 10 × 5670 + 4 × 5670
= 56700 + 22680
= 79380

So, the answer is ۷ answer here ۷ <answer>79380</answer>
